In [1]:
import math
import random
1455
4
43

# Game Constants
ROWS = 6
COLS = 7
EMPTY = 0
HUMAN_PIECE = 1
AI_PIECE = 2

WINDOW_LENGTH = 4
MAX_DEPTH = 4  # Increase for smarter AI, decrease for faster search


class ConnectFour:
    def __init__(self):
        # 6x7 grid initialized with zeros
        self.board = [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]

    def print_board(self):
        """Prints the board right-side up."""
        for r in range(ROWS - 1, -1, -1):
            print("| " + " ".join(str(self.board[r][c]) for c in range(COLS)) + " |")
        print("  " + " ".join(str(i) for i in range(COLS)))
        print("-" * 19)

    def is_valid_location(self, col):
        """Checks if top row of column is empty."""
        return self.board[ROWS - 1][col] == EMPTY

    def get_next_open_row(self, col):
        """Finds lowest empty row in selected column."""
        for r in range(ROWS):
            if self.board[r][col] == EMPTY:
                return r

    def drop_piece(self, row, col, piece):
        """Places disc in specified cell."""
        self.board[row][col] = piece

    def get_valid_locations(self):
        """Returns list of non-full columns."""
        return [c for c in range(COLS) if self.is_valid_location(c)]

    def winning_move(self, piece):
        """Checks if the given piece has 4-in-a-row anywhere on the board."""
        # Horizontal
        for c in range(COLS - 3):
            for r in range(ROWS):
                if all(self.board[r][c + i] == piece for i in range(4)):
                    return True

        # Vertical
        for c in range(COLS):
            for r in range(ROWS - 3):
                if all(self.board[r + i][c] == piece for i in range(4)):
                    return True

        # Positively sloped diagonals
        for c in range(COLS - 3):
            for r in range(ROWS - 3):
                if all(self.board[r + i][c + i] == piece for i in range(4)):
                    return True

        # Negatively sloped diagonals
        for c in range(COLS - 3):
            for r in range(3, ROWS):
                if all(self.board[r - i][c + i] == piece for i in range(4)):
                    return True

        return False

    def is_terminal_node(self):
        """Checks if game is over (win or draw)."""
        return (
            self.winning_move(HUMAN_PIECE)
            or self.winning_move(AI_PIECE)
            or len(self.get_valid_locations()) == 0
        )

    # ================= Evaluation Heuristic =================

    def evaluate_window(self, window, piece):
        """Scores a segment of 4 cells based on piece counts."""
        score = 0
        opp_piece = HUMAN_PIECE if piece == AI_PIECE else AI_PIECE

        if window.count(piece) == 4:
            score += 100
        elif window.count(piece) == 3 and window.count(EMPTY) == 1:
            score += 5
        elif window.count(piece) == 2 and window.count(EMPTY) == 2:
            score += 2

        if window.count(opp_piece) == 3 and window.count(EMPTY) == 1:
            score -= 4  # Block opponent's 3-in-a-row

        return score

    def score_position(self, piece):
        """Scores entire board state for specified piece."""
        score = 0

        # Preference to center column (strategic advantage)
        center_array = [self.board[r][COLS // 2] for r in range(ROWS)]
        center_count = center_array.count(piece)
        score += center_count * 3

        # Horizontal score
        for r in range(ROWS):
            row_array = self.board[r]
            for c in range(COLS - 3):
                window = row_array[c : c + WINDOW_LENGTH]
                score += self.evaluate_window(window, piece)

        # Vertical score
        for c in range(COLS):
            col_array = [self.board[r][c] for r in range(ROWS)]
            for r in range(ROWS - 3):
                window = col_array[r : r + WINDOW_LENGTH]
                score += self.evaluate_window(window, piece)

        # Positive Diagonal score
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                window = [self.board[r + i][c + i] for i in range(WINDOW_LENGTH)]
                score += self.evaluate_window(window, piece)

        # Negative Diagonal score
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                window = [self.board[r + 3 - i][c + i] for i in range(WINDOW_LENGTH)]
                score += self.evaluate_window(window, piece)

        return score

    # ================= Minimax with Alpha-Beta =================

    def minimax(self, depth, alpha, beta, maximizing_player):
        """Minimax recursion with Alpha-Beta Pruning."""
        valid_locations = self.get_valid_locations()
        is_terminal = self.is_terminal_node()

        if depth == 0 or is_terminal:
            if is_terminal:
                if self.winning_move(AI_PIECE):
                    return (None, 100000000000000)
                elif self.winning_move(HUMAN_PIECE):
                    return (None, -100000000000000)
                else:  # Game is draw
                    return (None, 0)
            else:  # Depth limit reached
                return (None, self.score_position(AI_PIECE))

        if maximizing_player:
            value = -math.inf
            best_col = random.choice(valid_locations)

            for col in valid_locations:
                row = self.get_next_open_row(col)
                self.drop_piece(row, col, AI_PIECE)

                _, new_score = self.minimax(depth - 1, alpha, beta, False)

                self.board[row][col] = EMPTY  # Backtrack

                if new_score > value:
                    value = new_score
                    best_col = col

                alpha = max(alpha, value)
                if alpha >= beta:
                    break  # Beta cut-off

            return best_col, value

        else:  # Minimizing player (Human)
            value = math.inf
            best_col = random.choice(valid_locations)

            for col in valid_locations:
                row = self.get_next_open_row(col)
                self.drop_piece(row, col, HUMAN_PIECE)

                _, new_score = self.minimax(depth - 1, alpha, beta, True)

                self.board[row][col] = EMPTY  # Backtrack

                if new_score < value:
                    value = new_score
                    best_col = col

                beta = min(beta, value)
                if alpha >= beta:
                    break  # Alpha cut-off

            return best_col, value

    def play(self):
        """Main game loop."""
        print("Welcome to Connect Four!")
        print("You are Player 1 (Piece 1). AI is Player 2 (Piece 2).")
        self.print_board()

        turn = random.choice([0, 1])  # Randomize starting player

        while not self.is_terminal_node():
            if turn == 0:
                # Human Turn
                try:
                    col = int(input(f"Enter column (0-{COLS-1}): "))
                    if not (0 <= col < COLS) or not self.is_valid_location(col):
                        print("Invalid column or column full. Try again.")
                        continue
                except ValueError:
                    print("Invalid input! Enter an integer.")
                    continue

                row = self.get_next_open_row(col)
                self.drop_piece(row, col, HUMAN_PIECE)

                if self.winning_move(HUMAN_PIECE):
                    self.print_board()
                    print("PLAYER 1 WINS!")
                    return
                turn = 1

            else:
                # AI Turn
                print("\nAI is calculating best move...")
                col, minimax_score = self.minimax(MAX_DEPTH, -math.inf, math.inf, True)

                if self.is_valid_location(col):
                    row = self.get_next_open_row(col)
                    self.drop_piece(row, col, AI_PIECE)

                    if self.winning_move(AI_PIECE):
                        self.print_board()
                        print("AI WINS!")
                        return
                    turn = 0

            self.print_board()

        print("GAME DRAW!")


# ================= Run Game =================
if __name__ == "__main__":
    game = ConnectFour()
    game.play()


Welcome to Connect Four!
You are Player 1 (Piece 1). AI is Player 2 (Piece 2).
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
  0 1 2 3 4 5 6
-------------------
Enter column (0-6): 6
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 1 |
  0 1 2 3 4 5 6
-------------------

AI is calculating best move...
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 2 0 0 1 |
  0 1 2 3 4 5 6
-------------------
Enter column (0-6): 1
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 1 0 2 0 0 1 |
  0 1 2 3 4 5 6
-------------------

AI is calculating best move...
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 1 0 2 0 0 1 |
  0 1 2 3 4 5 6
-------------------
Enter column (0-6): 11
Invalid column or column full. Try again.
Enter column (0-6): 1
| 0 0 